<a href="https://colab.research.google.com/github/Amarmurun0212/Diver/blob/main/PID0612.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Турбины тохируулах хаалтны PID коэффициентийн (Kp, Ki, Kd) машин сургалтын модель.
Dataset: turbine_pid_dataset.xlsx (sheet 'Dataset')

Энд оролт (X) нь системийн параметрүүд + хүссэн хариу үзүүлэлтүүд,
гарц (y) нь Kp, Ki, Kd байна. Өөрөөр хэлбэл: "ийм систем, ийм хариу
үзүүлэлт хүсвэл ямар PID coefficient тохируулах вэ?" гэсэн урвуу бодлого.
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

DATA_PATH = "turbine_pid_dataset.xlsx"

# 1. Өгөгдөл унших
df = pd.read_excel(DATA_PATH, sheet_name="Dataset")

FEATURES = [
    "Turbine_Inertia_J_kgm2",
    "Steam_Pressure_MPa",
    "Valve_Time_Const_s",
    "Load_Disturbance_pu",
    "Setpoint_Speed_rpm",
    "Initial_Speed_rpm",
    "Rise_Time_s",
    "Overshoot_percent",
    "Settling_Time_s",
    "Steady_State_Error_percent",
]
TARGETS = ["Kp", "Ki", "Kd"]

X = df[FEATURES]
y = df[TARGETS]

# 2. Train/test хуваах
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Стандартчилал
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 4. Модель сургах (RandomForest, multi-output)
model = MultiOutputRegressor(
    RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
)
model.fit(X_train_s, y_train)

# 5. Үнэлгэе
pred = model.predict(X_test_s)
for i, t in enumerate(TARGETS):
    mae = mean_absolute_error(y_test[t], pred[:, i])
    r2 = r2_score(y_test[t], pred[:, i])
    print(f"{t}: MAE={mae:.4f}, R2={r2:.4f}")

# 6. Модель ба scaler хадгалах
joblib.dump(model, "pid_model.joblib")
joblib.dump(scaler, "pid_scaler.joblib")
print("\nМодель хадгалагдлаа: pid_model.joblib, pid_scaler.joblib")


# 7. Predict функц - шинэ систем/нөхцөлд PID коэффициент таамаглах
def predict_pid(new_data: dict, model_path="pid_model.joblib", scaler_path="pid_scaler.joblib"):
    """
    new_data: dict, FEATURES-тэй адил түлхүүртэй
    Жишээ:
    {
        "Turbine_Inertia_J_kgm2": 250,
        "Steam_Pressure_MPa": 11,
        "Valve_Time_Const_s": 0.2,
        "Load_Disturbance_pu": 0.05,
        "Setpoint_Speed_rpm": 3000,
        "Initial_Speed_rpm": 2900,
        "Rise_Time_s": 0.8,
        "Overshoot_percent": 5,
        "Settling_Time_s": 3,
        "Steady_State_Error_percent": 0.5,
    }
    """
    mdl = joblib.load(model_path)
    scl = joblib.load(scaler_path)
    row = pd.DataFrame([new_data])[FEATURES]
    row_s = scl.transform(row)
    kp, ki, kd = mdl.predict(row_s)[0]
    return {"Kp": round(kp, 4), "Ki": round(ki, 4), "Kd": round(kd, 4)}


if __name__ == "__main__":
    example = {
        "Turbine_Inertia_J_kgm2": 250,
        "Steam_Pressure_MPa": 11,
        "Valve_Time_Const_s": 0.2,
        "Load_Disturbance_pu": 0.05,
        "Setpoint_Speed_rpm": 3000,
        "Initial_Speed_rpm": 2900,
        "Rise_Time_s": 0.8,
        "Overshoot_percent": 5,
        "Settling_Time_s": 3,
        "Steady_State_Error_percent": 0.5,
    }
    print("\nТаамаглал:", predict_pid(example))

Kp: MAE=0.7819, R2=0.8788
Ki: MAE=0.5284, R2=0.7648
Kd: MAE=0.1614, R2=0.8588

Модель хадгалагдлаа: pid_model.joblib, pid_scaler.joblib

Таамаглал: {'Kp': np.float64(5.3959), 'Ki': np.float64(0.1571), 'Kd': np.float64(1.6385)}
